### 2.0 Setup

In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent
src_path = root/'src'
sys.path.insert(0,str(src_path))
import numpy as np
import matplotlib.pyplot as plt
import wandb
from utils.data_loader import load_fashion_mnist,load_mnist
from ann.neural_network import NeuralNetwork

In [2]:
PROJECT_NAME = "DA6401-Assignment-1"

In [3]:
class MockArgs:
    """A simple class to mimic the argspace namespace."""
    pass

def sweep_train():
    wandb.init()
    config = wandb.config

    args = MockArgs()
    args.loss = config.loss
    args.learning_rate = config.learning_rate
    args.weight_decay = config.weight_decay
    args.optimizer = config.optimizer
    args.activation = config.activation
    args.weight_init = config.weight_init
    args.hidden_size = [int(x) for x in config.layer_config.split('_')]
    args.num_layers = len([int(x) for x in config.layer_config.split('_')])
    x_train,y_train,x_test,y_test = load_mnist()
    model = NeuralNetwork(args)
    model.train(x_train, y_train, config.epochs, config.batch_size)

    wandb.finish()

### 2.1 Data Exploration and Class Distribution

In [6]:
wandb.init(project= PROJECT_NAME, name= 'question_2.1')
print('loading MINST dataset')
x_train,y_train,x_test,y_test = load_mnist()
columns = ["Class Label"] + [f"Sample {i+1}" for i in range(5)]
table = wandb.Table(columns=columns)
print("collecting samples")
for label in range(10):
    samples = np.where(y_train == label)[0]
    first5samples = samples[:5]
    row_data = [str(label)]
    for sample in first5samples:
        flat_img = x_train[sample]
        img = flat_img.reshape(28,28) * 255
        row_data.append(wandb.Image(img,caption=f"{label}"))
    table.add_data(*row_data)
wandb.log({
    "sample_data": table
})
wandb.finish()
print("finished logging samples to wandb")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.
wandb: Currently logged in as: cs25m034 (cs25m034-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


loading MINST dataset
collecting samples


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


finished logging samples to wandb


### 2.2 Hyperparameter sweep

In [4]:
layer_configs = ['128', '96', '64', '48', '32', '16',
                '128_128', '64_64', '32_32',
                '128_96', '128_64', '128_32',               
                '96_64', '64_32', '64_16', '32_16',
                '128_128_128', '64_64_64',                 
                '128_96_64', '128_64_32', '64_48_32',       
                '128_64_16', '128_32_16', '64_32_16',     
                '64_128_64', '32_64_32',
                '128_128_128_128', '64_64_64_64',           
                '128_128_64_64', '128_64_64_32',
                '128_96_64_32', '128_64_32_16', '64_32_16_8',
                '128_128_128_128_128', '64_64_64_64_64',
                '128_128_64_32_16', '128_64_32_16_8', 
                '64_64_32_32_16',
                '128_128_128_128_128_128',
                '64_64_64_64_64_64',
                '128_128_64_64_32_32',
                '128_96_64_48_32_16',
                '128_64_32_16_8_4'
                ]
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'best_val_accuracy_till_now',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {'values': [0.1, 0.01, 0.001, 0.0001]},
        'batch_size': {'values': [32, 64, 128, 256]},
        'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop']},
        'activation': {'values': ['relu', 'tanh', 'sigmoid']},
        'weight_init': {'values': ['random', 'xavier']},
        'weight_decay': {'values': [0.0001, 0.001, 0.01, 0]},
        'layer_config': {'values': layer_configs},
        'epochs': {'values': [2,4,6,8,10,12,15,20]},
        'loss': {'values': ['mean_squared_error', 'cross_entropy']}
    }
}

In [5]:
sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train, count=100)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Create sweep with ID: qmpmphdm
Sweep URL: https://wandb.ai/cs25m034-indian-institute-of-technology-madras/DA6401-Assignment-1/sweeps/qmpmphdm


wandb: Agent Starting Run: s86zfb11 with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	layer_config: 128
wandb: 	learning_rate: 0.1
wandb: 	loss: mean_squared_error
wandb: 	optimizer: nag
wandb: 	weight_decay: 0
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.
wandb: Currently logged in as: cs25m034 (cs25m034-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/15 - Train Loss: 0.0904 - Val Acc: 0.1095
Epoch 2/15 - Train Loss: 0.0903 - Val Acc: 0.1022
Epoch 3/15 - Train Loss: 0.0902 - Val Acc: 0.0998
Epoch 4/15 - Train Loss: 0.0904 - Val Acc: 0.1040
Epoch 5/15 - Train Loss: 0.0904 - Val Acc: 0.0998
Epoch 6/15 - Train Loss: 0.0901 - Val Acc: 0.0983
Epoch 7/15 - Train Loss: 0.0904 - Val Acc: 0.1040
Epoch 8/15 - Train Loss: 0.0902 - Val Acc: 0.0993
Epoch 9/15 - Train Loss: 0.0902 - Val Acc: 0.1022
Epoch 10/15 - Train Loss: 0.0903 - Val Acc: 0.0998
Epoch 11/15 - Train Loss: 0.0903 - Val Acc: 0.1100
Epoch 12/15 - Train Loss: 0.0902 - Val Acc: 0.1003
Epoch 13/15 - Train Loss: 0.0901 - Val Acc: 0.0998
Epoch 14/15 - Train Loss: 0.0902 - Val Acc: 0.0978


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


...

wandb: Agent Starting Run: jpq6wx6f with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 32
wandb: 	epochs: 2
wandb: 	layer_config: 32
wandb: 	learning_rate: 0.1
wandb: 	loss: cross_entropy
wandb: 	optimizer: nag
wandb: 	weight_decay: 0.001
wandb: 	weight_init: random
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Epoch 1/2 - Train Loss: 0.4507 - Val Acc: 0.9327


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 2/2 - Train Loss: 0.4226 - Val Acc: 0.9328


best_val_accuracy_till_now,▁█
epoch,▁█
first_layer_grad_norm,█▁
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁█
neuron_0_grad,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇██
neuron_1_grad,██████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁
neuron_2_grad,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇██
neuron_3_grad,███████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁
neuron_4_grad,███████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
+4,...


### 2.3 Optimizer showdown

In [7]:
sweep_config = {
    'method': 'grid',
    'parameters': {
        'learning_rate': {'values': [0.001]},
        'batch_size': {'values': [128]},
        'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop']},
        'activation': {'values': ['relu']},
        'weight_init': {'values': ['xavier']},
        'weight_decay': {'values': [0.0001]},
        'layer_config': {'values': ['128_128_128']},
        'epochs': {'values': [5]},
        'loss': {'values': ['cross_entropy']}
    }
}

sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train)

Create sweep with ID: pw0gcx28
Sweep URL: https://wandb.ai/cs25m034-indian-institute-of-technology-madras/DA6401-Assignment-1/sweeps/pw0gcx28


wandb: Agent Starting Run: ziswjmjf with config:
wandb: 	activation: relu
wandb: 	batch_size: 128
wandb: 	epochs: 5
wandb: 	layer_config: 128_128_128
wandb: 	learning_rate: 0.001
wandb: 	loss: cross_entropy
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0.0001
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Epoch 1/5 - Train Loss: 2.2209 - Val Acc: 0.2878
Epoch 2/5 - Train Loss: 2.0798 - Val Acc: 0.4397
Epoch 3/5 - Train Loss: 1.8922 - Val Acc: 0.5583
Epoch 4/5 - Train Loss: 1.6553 - Val Acc: 0.6413


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 1.3969 - Val Acc: 0.7058


...

Epoch 1/5 - Train Loss: 0.1674 - Val Acc: 0.9567
Epoch 2/5 - Train Loss: 0.1217 - Val Acc: 0.9625
Epoch 3/5 - Train Loss: 0.0961 - Val Acc: 0.9660
Epoch 4/5 - Train Loss: 0.0732 - Val Acc: 0.9727


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.0688 - Val Acc: 0.9742


best_val_accuracy_till_now,▁▃▅▇█
epoch,▁▃▅▆█
first_layer_grad_norm,▅█▅▅▁
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▆▆██
layer:1_dead_neuron_pct,▁▄▅▇█
layer:2_dead_neuron_pct,▁▅▇██
neuron_0_grad,▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
neuron_1_grad,█▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad,▁▃▅▆▆███████████████████████████████████
+6,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


### 2.4 Vanishing gradient analysis

In [8]:
sweep_config = {
    'method': 'grid',
    'parameters': {
        'learning_rate': {'values': [0.001]},
        'batch_size': {'values': [128]},
        'optimizer': {'values': ['rmsprop']},
        'activation': {'values': ['sigmoid','relu']},
        'weight_init': {'values': ['xavier']},
        'weight_decay': {'values': [0.0001]},
        'layer_config': {'values': ['128_128_128','128_64_32','128_128',"128_64_32_16"]},
        'epochs': {'values': [5]},
        'loss': {'values': ['cross_entropy']}
    }
}

sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train)

Create sweep with ID: bkrajltw
Sweep URL: https://wandb.ai/cs25m034-indian-institute-of-technology-madras/DA6401-Assignment-1/sweeps/bkrajltw


wandb: Agent Starting Run: t76di8e9 with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 128
wandb: 	epochs: 5
wandb: 	layer_config: 128_128_128
wandb: 	learning_rate: 0.001
wandb: 	loss: cross_entropy
wandb: 	optimizer: rmsprop
wandb: 	weight_decay: 0.0001
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Epoch 1/5 - Train Loss: 0.4124 - Val Acc: 0.8933
Epoch 2/5 - Train Loss: 0.3122 - Val Acc: 0.9260
Epoch 3/5 - Train Loss: 0.2528 - Val Acc: 0.9427
Epoch 4/5 - Train Loss: 0.2395 - Val Acc: 0.9457


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.2149 - Val Acc: 0.9528


...

Epoch 1/5 - Train Loss: 0.2140 - Val Acc: 0.9407
Epoch 2/5 - Train Loss: 0.1367 - Val Acc: 0.9610
Epoch 3/5 - Train Loss: 0.1007 - Val Acc: 0.9700
Epoch 4/5 - Train Loss: 0.0990 - Val Acc: 0.9637


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.0715 - Val Acc: 0.9730


best_val_accuracy_till_now,▁▅▇▇█
epoch,▁▃▅▆█
first_layer_grad_norm,▅▁█▂▁
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▄▇▇█
layer:1_dead_neuron_pct,▁▃███
layer:2_dead_neuron_pct,▁▅▅▇█
layer:3_dead_neuron_pct,▁▃▁▂█
neuron_0_grad,█▇▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
neuron_1_grad,█▆▅▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+7,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


### 2.5 Dead neuron investigation

In [9]:
sweep_config = {
    'method': 'grid',
    'parameters': {
        'learning_rate': {'values': [0.1]},
        'batch_size': {'values': [128]},
        'optimizer': {'values': ['sgd']},
        'activation': {'values': ['relu','tanh']},
        'weight_init': {'values': ['xavier']},
        'weight_decay': {'values': [0]},
        'layer_config': {'values': ['128_128_128']},
        'epochs': {'values': [15]},
        'loss': {'values': ['cross_entropy']}
    }
}

sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train)

Create sweep with ID: 1t44gmqi
Sweep URL: https://wandb.ai/cs25m034-indian-institute-of-technology-madras/DA6401-Assignment-1/sweeps/1t44gmqi


wandb: Agent Starting Run: km39n1oe with config:
wandb: 	activation: relu
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	layer_config: 128_128_128
wandb: 	learning_rate: 0.1
wandb: 	loss: cross_entropy
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Epoch 1/15 - Train Loss: 0.2280 - Val Acc: 0.9343
Epoch 2/15 - Train Loss: 0.1548 - Val Acc: 0.9538
Epoch 3/15 - Train Loss: 0.1176 - Val Acc: 0.9608
Epoch 4/15 - Train Loss: 0.1031 - Val Acc: 0.9625
Epoch 5/15 - Train Loss: 0.0781 - Val Acc: 0.9673
Epoch 6/15 - Train Loss: 0.0905 - Val Acc: 0.9618
Epoch 7/15 - Train Loss: 0.0517 - Val Acc: 0.9728
Epoch 8/15 - Train Loss: 0.0456 - Val Acc: 0.9718
Epoch 9/15 - Train Loss: 0.0404 - Val Acc: 0.9738
Epoch 10/15 - Train Loss: 0.0387 - Val Acc: 0.9727
Epoch 11/15 - Train Loss: 0.0267 - Val Acc: 0.9783
Epoch 12/15 - Train Loss: 0.0268 - Val Acc: 0.9768
Epoch 13/15 - Train Loss: 0.0338 - Val Acc: 0.9727
Epoch 14/15 - Train Loss: 0.0199 - Val Acc: 0.9773


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.0187 - Val Acc: 0.9773


...

Epoch 1/15 - Train Loss: 0.2641 - Val Acc: 0.9253
Epoch 2/15 - Train Loss: 0.2000 - Val Acc: 0.9407
Epoch 3/15 - Train Loss: 0.1610 - Val Acc: 0.9523
Epoch 4/15 - Train Loss: 0.1357 - Val Acc: 0.9570
Epoch 5/15 - Train Loss: 0.1073 - Val Acc: 0.9630
Epoch 6/15 - Train Loss: 0.0933 - Val Acc: 0.9665
Epoch 7/15 - Train Loss: 0.0786 - Val Acc: 0.9715
Epoch 8/15 - Train Loss: 0.0721 - Val Acc: 0.9702
Epoch 9/15 - Train Loss: 0.0637 - Val Acc: 0.9735
Epoch 10/15 - Train Loss: 0.0563 - Val Acc: 0.9737
Epoch 11/15 - Train Loss: 0.0520 - Val Acc: 0.9748
Epoch 12/15 - Train Loss: 0.0438 - Val Acc: 0.9752
Epoch 13/15 - Train Loss: 0.0456 - Val Acc: 0.9730
Epoch 14/15 - Train Loss: 0.0351 - Val Acc: 0.9778


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.0342 - Val Acc: 0.9758


best_val_accuracy_till_now,▁▃▅▅▆▆▇▇▇▇█████
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
first_layer_grad_norm,█▇▅▅▅▆▃▄▅▁▅▇█▁▂
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▁▂▃▃▄▅▅▆▆▇▇▇██
layer:1_dead_neuron_pct,▁▂▂▃▃▄▅▅▅▆▆▇▇▇█
layer:2_dead_neuron_pct,▁▁▃▃▄▄▅▅▆▆▆▇▇▇█
neuron_0_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+6,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


### 2.6 Loss function comparsion

In [10]:
sweep_config = {
    'method': 'grid',
    'parameters': {
        'learning_rate': {'values': [0.001]},
        'batch_size': {'values': [128]},
        'optimizer': {'values': ['sgd']},
        'activation': {'values': ['relu']},
        'weight_init': {'values': ['xavier']},
        'weight_decay': {'values': [0]},
        'layer_config': {'values': ['128_128_128']},
        'epochs': {'values': [15]},
        'loss': {'values': ['mean_squared_error','cross_entropy']}
    }
}

sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train)

Create sweep with ID: h7dbc0hd
Sweep URL: https://wandb.ai/cs25m034-indian-institute-of-technology-madras/DA6401-Assignment-1/sweeps/h7dbc0hd


wandb: Agent Starting Run: tppuu4en with config:
wandb: 	activation: relu
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	layer_config: 128_128_128
wandb: 	learning_rate: 0.001
wandb: 	loss: mean_squared_error
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.


Epoch 1/15 - Train Loss: 0.0851 - Val Acc: 0.3148
Epoch 2/15 - Train Loss: 0.0761 - Val Acc: 0.4728
Epoch 3/15 - Train Loss: 0.0700 - Val Acc: 0.5652
Epoch 4/15 - Train Loss: 0.0654 - Val Acc: 0.6180
Epoch 5/15 - Train Loss: 0.0618 - Val Acc: 0.6528
Epoch 6/15 - Train Loss: 0.0588 - Val Acc: 0.6855
Epoch 7/15 - Train Loss: 0.0562 - Val Acc: 0.7075
Epoch 8/15 - Train Loss: 0.0540 - Val Acc: 0.7270
Epoch 9/15 - Train Loss: 0.0520 - Val Acc: 0.7422
Epoch 10/15 - Train Loss: 0.0503 - Val Acc: 0.7583
Epoch 11/15 - Train Loss: 0.0487 - Val Acc: 0.7710
Epoch 12/15 - Train Loss: 0.0472 - Val Acc: 0.7818
Epoch 13/15 - Train Loss: 0.0459 - Val Acc: 0.7933
Epoch 14/15 - Train Loss: 0.0446 - Val Acc: 0.8032


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.0435 - Val Acc: 0.8140


...

Epoch 1/15 - Train Loss: 2.2330 - Val Acc: 0.1695
Epoch 2/15 - Train Loss: 2.1106 - Val Acc: 0.3245
Epoch 3/15 - Train Loss: 1.9513 - Val Acc: 0.5033
Epoch 4/15 - Train Loss: 1.7377 - Val Acc: 0.6133
Epoch 5/15 - Train Loss: 1.4818 - Val Acc: 0.7005
Epoch 6/15 - Train Loss: 1.2261 - Val Acc: 0.7650
Epoch 7/15 - Train Loss: 1.0118 - Val Acc: 0.7975
Epoch 8/15 - Train Loss: 0.8544 - Val Acc: 0.8165
Epoch 9/15 - Train Loss: 0.7443 - Val Acc: 0.8297
Epoch 10/15 - Train Loss: 0.6665 - Val Acc: 0.8417
Epoch 11/15 - Train Loss: 0.6098 - Val Acc: 0.8502
Epoch 12/15 - Train Loss: 0.5668 - Val Acc: 0.8562
Epoch 13/15 - Train Loss: 0.5329 - Val Acc: 0.8640
Epoch 14/15 - Train Loss: 0.5059 - Val Acc: 0.8700


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.4838 - Val Acc: 0.8742


best_val_accuracy_till_now,▁▃▄▅▆▇▇▇███████
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
first_layer_grad_norm,▁▃▇▆▅▇▄█▆▃▅▇█▂▆
iteration_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,██▇▆▅▄▃▃▂▂▂▁▁▁▁
layer:1_dead_neuron_pct,██▇▆▅▄▃▃▂▂▂▁▁▁▁
layer:2_dead_neuron_pct,█▇▆▅▄▃▃▂▂▂▁▁▁▁▁
neuron_0_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+6,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


### 2.8 Confusion matrix

In [6]:
from inference import cmatrix
x_train,y_train,x_test,y_test = load_mnist()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.8_confusion_matrix",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 10,
        "num_layers": 2,
        "hidden_size": [128, 128],
        "activation": "relu",
        "loss": 'cross_entropy'
    }
)

args = MockArgs()
args.dataset = 'mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.001
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'relu'
args.weight_init = 'xavier'
args.num_layers = 2
args.hidden_size = [128, 128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 10,batch_size= 128)
ypred,ytest = cmatrix(model,x_test,y_test)
wandb.log({'confusion_matrix': wandb.plot.confusion_matrix(preds=ypred.flatten(), y_true=ytest.flatten(), class_names=[str(i) for i in range(10)])})

wandb.finish()

Epoch 1/10 - Train Loss: 0.1506 - Val Acc: 0.9535
Epoch 2/10 - Train Loss: 0.0928 - Val Acc: 0.9642
Epoch 3/10 - Train Loss: 0.0727 - Val Acc: 0.9697
Epoch 4/10 - Train Loss: 0.0493 - Val Acc: 0.9740
Epoch 5/10 - Train Loss: 0.0353 - Val Acc: 0.9760
Epoch 6/10 - Train Loss: 0.0286 - Val Acc: 0.9772
Epoch 7/10 - Train Loss: 0.0281 - Val Acc: 0.9782
Epoch 8/10 - Train Loss: 0.0179 - Val Acc: 0.9802
Epoch 9/10 - Train Loss: 0.0194 - Val Acc: 0.9767
Epoch 10/10 - Train Loss: 0.0125 - Val Acc: 0.9785


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_accuracy_till_now,▁▄▅▆▇▇▇███
epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,▆▄█▆▅▂▆▄▁▃
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▄▄▆▆▇▇███
layer:1_dead_neuron_pct,▁▅▃▆▆▇▇█▇█
neuron_0_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_3_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


### 2.9 weight initialzation and symmetry

In [4]:
x_train,y_train,x_test,y_test = load_mnist()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.9_zeros_init",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 5,
        "num_layers": 3,
        "hidden_size": [128, 128, 128],
        "activation": "sigmoid",
        "loss": 'cross_entropy',
        "weight_init": "zeros"
    }
)

args = MockArgs()
args.dataset = 'mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.001
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'sigmoid'
args.weight_init = 'zeros'
args.num_layers = 3
args.hidden_size = [128, 128, 128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 5,batch_size= 128)

wandb.finish()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.9_xavier_init",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 5,
        "num_layers": 3,
        "hidden_size": [128, 128, 128],
        "activation": "sigmoid",
        "loss": 'cross_entropy',
        "weight_init": "xavier"
    }
)

args = MockArgs()
args.dataset = 'mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.001
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'sigmoid'
args.weight_init = 'xavier'
args.num_layers = 3
args.hidden_size = [128, 128, 128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 5,batch_size= 128)

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.
wandb: Currently logged in as: cs25m034 (cs25m034-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/5 - Train Loss: 1.9724 - Val Acc: 0.2142
Epoch 2/5 - Train Loss: 1.8887 - Val Acc: 0.2122
Epoch 3/5 - Train Loss: 1.8422 - Val Acc: 0.2295
Epoch 4/5 - Train Loss: 1.7852 - Val Acc: 0.3383


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 1.7295 - Val Acc: 0.3413


best_val_accuracy_till_now,▁▁▂██
epoch,▁▃▅▆█
first_layer_grad_norm,▁▂▆█▆
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
layer:0_dead_neuron_pct,█▇▅▂▁
layer:1_dead_neuron_pct,█▁▁▁▁
layer:2_dead_neuron_pct,▇█▄▄▁
neuron_0_grad,▄▄▄▄▄▅▄▄▄▄▅▄▅▅▅▅▄▃▅▆▅▅▅▅▃▆▄▆▄▆▄▆▅▅▄██▇▁▄
neuron_1_grad,▂▂▂▂▂▂▃▂▂▂▃▃▂▄▃▂▃▃▁▄▃▃▄▄▃▇▆▂▄▂▄▃▂▅▄▃██▆▂
neuron_2_grad,▄▄▄▄▄▄▄▄▄▄▄▅▅▅▄▄▃▅▆▅▅▅▅▃▇▄▆▄▆▆▄▆▅▅▄██▇▁▄
+6,...


Epoch 1/5 - Train Loss: 0.3315 - Val Acc: 0.9048
Epoch 2/5 - Train Loss: 0.2277 - Val Acc: 0.9307
Epoch 3/5 - Train Loss: 0.1710 - Val Acc: 0.9467
Epoch 4/5 - Train Loss: 0.1502 - Val Acc: 0.9495


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.1134 - Val Acc: 0.9635


best_val_accuracy_till_now,▁▄▆▆█
epoch,▁▃▅▆█
first_layer_grad_norm,▁▃█▆▇
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▃▅▇█
layer:1_dead_neuron_pct,▁▃▇▇█
layer:2_dead_neuron_pct,▁▄▆▇█
neuron_0_grad,▃▃▃▃▃▃▃▃▃▂▃▄▄▂▃▂▃▃▃▃▂▃▄▃▃▃▃▃▃▄▂▃▅▃▅▃▁▂█▄
neuron_1_grad,▅▅▅▅▆▅▅▄▆▅▅▆▅▅▇▅▅▆▅▅▅▅▅▆▄▆▅▄▆▆▆▅▅▇▅█▅▅▁▃
neuron_2_grad,▃▂▅▃▂▆▃▃▅▄▅▄▃▆▁▅▇▃▆▃▄█▄▂▇▂█▆▅▁▆▂▅▁▃▃▅▆▂▁
+6,...


### 2.10 Fashion MNIST challenge

In [ ]:
x_train,y_train,x_test,y_test = load_fashion_mnist()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.10_config_1",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 10,
        "num_layers": 2,
        "hidden_size": [128, 128],
        "activation": "relu",
        "loss": 'cross_entropy',
        "weight_init": "xavier"
    }
)

args = MockArgs()
args.dataset = 'fashion_mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.001
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'relu'
args.weight_init = 'xavier'
args.num_layers = 2
args.hidden_size = [128, 128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 10,batch_size= 128)

wandb.finish()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.10_config_2",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 8,
        "num_layers": 1,
        "hidden_size": [128],
        "activation": "sigmoid",
        "loss": 'cross_entropy',
        "weight_init": "random"
    }
)

args = MockArgs()
args.dataset = 'fashion_mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.01
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'sigmoid'
args.weight_init = 'random'
args.num_layers = 1
args.hidden_size = [128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 8,batch_size= 256)

wandb.finish()

wandb.init(
    project = PROJECT_NAME,
    name = f"2.10_config_3",
    config = {
        "optimizer": 'rmsprop',
        "epochs": 8,
        "num_layers": 1,
        "hidden_size": [128],
        "activation": "relu",
        "loss": 'cross_entropy',
        "weight_init": "random"
    }
)

args = MockArgs()
args.dataset = 'fashion_mnist'
args.loss = 'cross_entropy'
args.learning_rate = 0.01
args.weight_decay = 0.0
args.optimizer = 'rmsprop'
args.activation = 'relu'
args.weight_init = 'random'
args.num_layers = 1
args.hidden_size = [128]

model = NeuralNetwork(args)
model.train(x_train, y_train, epochs = 8,batch_size= 256)

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/srihasa/.netrc.
wandb: Currently logged in as: cs25m034 (cs25m034-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/10 - Train Loss: 0.4584 - Val Acc: 0.8230
Epoch 2/10 - Train Loss: 0.4023 - Val Acc: 0.8408
Epoch 3/10 - Train Loss: 0.3195 - Val Acc: 0.8745
Epoch 4/10 - Train Loss: 0.3034 - Val Acc: 0.8695
Epoch 5/10 - Train Loss: 0.3045 - Val Acc: 0.8710
Epoch 6/10 - Train Loss: 0.2963 - Val Acc: 0.8723
Epoch 7/10 - Train Loss: 0.2464 - Val Acc: 0.8827
Epoch 8/10 - Train Loss: 0.2781 - Val Acc: 0.8747
Epoch 9/10 - Train Loss: 0.2782 - Val Acc: 0.8633


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 10/10 - Train Loss: 0.2129 - Val Acc: 0.8898


best_val_accuracy_till_now,▁▃▆▆▆▆▇▇▇█
epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,█▂▁▄▃▁▁▆█▂
iteration_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▅▇█▇█▇▇██
layer:1_dead_neuron_pct,▁▃▃▅▄▇▇▆█▇
neuron_0_grad,▃▆▁▇▆▃▄▄▆▄▃▆▃▄▄▃▄▃▇▃▃▅▄▅▂▄▂▅▅▄▂█▂▄▄▄▅▄▄▄
neuron_1_grad,█▃▃▃▃▃▃▄▃▃▃▃▃▃▃▄▃▃▃▃▃▃▃▃▃▃▃▃▃▄▂▂▃▅▄▁▅▂▃▁
neuron_2_grad,▅▃▁█▂▆▃▄▃▃▃▃▃▃▅▃▃▆▂▂▄▃▄▁▆▁▃▂▄▂▁▆▃▃▄▄▄▄▂▃
neuron_3_grad,▅▅▅▄▅▄▅▄▅▄▆▄▄▄▄▄▅▃▃█▅▅▄▄▅▄▃▇▃▁▂▅▄▄▆▃▆▂▇▆
+5,...


...

best_val_accuracy_till_now,▁▅▇▇████
epoch,▁▂▃▄▅▆▇█
first_layer_grad_norm,▆▄▂▂▅▄█▁
iteration_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▄▅▆▇▇██
neuron_0_grad,█▄▄▃▃▅▄▃▆▁▂▇▂▆▃▃▅▄▅▅▆▄▅▅▃▃▅▅▅▄▄▅▅▄▄▂▅▄▄▅
neuron_1_grad,█▂▂▃▁▂▃▃▂▃▃▃▃▃▃▄▂▃▄▂▃▂▄▂▄▃▂▄▁▃▃▃▃▃▃▄▂▄▂▃
neuron_2_grad,▁▅▅▅██▂▂▇▁▃▅▃▄▃▃▃▅▃▅▄▅▃▃▄▄▄▄▄▃▄▄▄▄▄▃▄▃▄▄
neuron_3_grad,▃█▃▅▅▁▇▅▄▅▅▅▅▅▄▄▆▃▆▄▄▅▅▃▅▅▅▅▄▅▆▄▆▄▅▃▆▄▅▅
neuron_4_grad,█▂█▄▁▁▄▆▄▄▄▄▄▅▅▃▁▆▃▄▅▂▆▃▅▃▄▅▃▅▆▃▄▄▄▃▄▅▃▃
+4,...


Epoch 1/8 - Train Loss: 0.5040 - Val Acc: 0.8187
Epoch 2/8 - Train Loss: 0.4788 - Val Acc: 0.8047
Epoch 3/8 - Train Loss: 0.4164 - Val Acc: 0.8325
Epoch 4/8 - Train Loss: 0.3940 - Val Acc: 0.8393
Epoch 5/8 - Train Loss: 0.3282 - Val Acc: 0.8613
Epoch 6/8 - Train Loss: 0.3767 - Val Acc: 0.8382
Epoch 7/8 - Train Loss: 0.4399 - Val Acc: 0.8383


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 8/8 - Train Loss: 0.2946 - Val Acc: 0.8697


best_val_accuracy_till_now,▁▁▃▄▇▇▇█
epoch,▁▂▃▄▅▆▇█
first_layer_grad_norm,▆▇▆▅▁▆█▁
iteration_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
layer:0_dead_neuron_pct,▁▇▄▇▇▇▄█
neuron_0_grad,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad,█▁▂▂▁▂▂▁▂▂▂▂▁▂▁▂▂▂▁▂▁▁▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂
neuron_2_grad,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_3_grad,▁▄▁█▇▇▃█▄▂▃▆▂▆▄▇▄▄▄▄▄▅▄▅▄▆▄▄▄▄▄▄▄▄▄▄▄▄▄▄
neuron_4_grad,▁▆█▇▁▆▆▆▄▅▄▅▄█▃▃▆▅▅▆▆▄▅▅▅▃▅▅▄▇▅▅▅▆▄▄▆▅▅▅
+4,...
